# Step 1: Install Libraries

In [ ]:
#!pip install pandas
#!pip install pyarrow fastparquet
#!pip install polars
#!pip install duckdb
#pip install pyspark
# !pip install dask
# !pip install great_expectations
#!pip install pandera pandas
#!pip install pydantic pandas
#!pip install pyjanitor pandas ipython
#!pip install toolz

Defaulting to user installation because normal site-packages is not writeable


# Step 2: Import Libraries

In [20]:
#----pandas-----
import os
import pandas as pd
from IPython.display import FileLink, display
#----pyarrow-----
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.parquet as pq
import time
import os
import psutil
import json
#----polars-----
import polars as pl
#-----DuckDB-----
import duckdb
#-----Pyspark-----
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, regexp_replace
#-----Dask-----
import dask.dataframe as dd
#-----Great Expectations-----
import great_expectations as gx
#-----Pandera------
import pandera as pa
#-----pydantic------
from pydantic import BaseModel, ValidationError, field_validator
#-----pyjanitor-----
import janitor
#-----toolz-----
from toolz import pipe

# Step 3: Import Data

In [ ]:
# Importing the sample data from csv file
df = pd.read_csv("../Input_file/sample_500_rows.csv")

# Print dataframe (few rows)
df.head(10)
df.tail(10)
print('----------------------')

# Print the dataframe info
print('Info: \n',df.info())
print('----------------------')

# Print the dataframe details 
print('Describe:',df.describe())
print('----------------------')

# Print 
print('No. of rows & columns',df.shape)

----------------------
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        500 non-null    int64
 1   name      500 non-null    str  
 2   category  500 non-null    str  
 3   price     356 non-null    str  
 4   quantity  329 non-null    str  
 5   date      264 non-null    str  
dtypes: int64(1), str(5)
memory usage: 37.4 KB
Info: 
 None
----------------------
Describe:                id
count  500.000000
mean   250.500000
std    144.481833
min      1.000000
25%    125.750000
50%    250.500000
75%    375.250000
max    500.000000
----------------------
No. of rows & columns (500, 6)


# Step 4: Data Cleaning and Transformation Pipeline

In [11]:
# Remove exact duplicates
df = df.drop_duplicates()

# Clean price column
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Remove invalid price rows
df = df.dropna(subset=["price"])

# Remove zero or negative prices (bad data)
df = df[df["price"] > 0]

# Remove missing category (cannot group without it)
df = df.dropna(subset=["category"])

# remove empty strings in category
df = df[df["category"] != ""]

# Transform
summary = df.groupby("category")["price"].mean()

print(summary)

category
Clothing       1286.898734
Electronics    1421.465517
Grocery        1390.403846
Name: price, dtype: float64


# Step 5: Libraries Exploration

## Step 5.1: CORE DATA PROCESSING

### Step 5.1.1: Pandas

In [19]:
def analyze_data(df):
    """
    Perform basic analysis on a product dataset and save summary results.

    This function computes:
    - Average price per category
    - Count of products per category
    - Maximum price per category
    - Minimum price per category
    - Top 5 most expensive products

    It then creates a summary DataFrame (avg, max, min price per category),
    saves it as a CSV file, and returns a clickable file link (for Jupyter).

    Parameters:
    ----------
    df : pandas.DataFrame
        Input DataFrame expected to contain at least:
        - 'category' column
        - 'price' column

    Returns:
    -------
    IPython.display.FileLink
        Clickable link to the generated CSV file
    """

    # Store all computed results in a dictionary
    results = {
        # Average price per category
        "avg_price": df.groupby("category")["price"].mean(),

        # Count of products per category
        "count_products": df["category"].value_counts(),

        # Maximum price per category
        "max_price": df.groupby("category")["price"].max(),

        # Minimum price per category
        "min_price": df.groupby("category")["price"].min(),

        # Top 5 highest priced products
        "top_5_products": df.sort_values(by="price", ascending=False).head(5)
    }
    
    # Ensure output directory exists (creates if not present)
    os.makedirs("../output_file", exist_ok=True)
    
    # Combine selected metrics into a summary DataFrame
    summary_df = pd.concat([
        results["avg_price"],
        results["max_price"],
        results["min_price"]
    ], axis=1)

    # Rename columns for clarity
    summary_df.columns = ["avg_price", "max_price", "min_price"]
    
    # Define output file path
    output_path = "../output_file/pandas_output_file.csv"

    # Save summary DataFrame to CSV
    summary_df.to_csv(output_path, index=True)

    # Return clickable link (works in Jupyter Notebook)
    return FileLink(output_path)


# Call function
analyze_data(df)

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\pandas_output_file.csv

### Step 5.1.2: pyarrow

In [26]:
def columnar_storage(df):
    """
    Convert a Pandas DataFrame into columnar format (Parquet),
    save it to disk, and return a preview along with a download link.

    This function performs the following steps:
    1. Records the input DataFrame shape
    2. Converts the DataFrame into a PyArrow Table (columnar format)
    3. Saves the data as a Parquet file
    4. Reads the file back into Pandas for verification/preview

    Parameters:
    ----------
    df : pandas.DataFrame
        Input DataFrame to be converted into columnar storage format

    Returns:
    -------
    tuple
        - IPython.display.FileLink : Clickable link to the saved Parquet file
        - pandas.DataFrame : Preview (first 5 rows) of the stored data
    """

    # Dictionary to store intermediate results/metadata
    results = {}

    # Step 1: Capture input DataFrame shape (rows, columns)
    results["input_shape"] = df.shape

    # Ensure output directory exists (creates if missing)
    os.makedirs("../output_file", exist_ok=True)

    # Step 2: Convert Pandas DataFrame to PyArrow Table (columnar format)
    table = pa.Table.from_pandas(df)

    # Store schema information of the columnar table
    results["columnar_schema"] = table.schema

    # Step 3: Save the table as a Parquet file
    output_path = "../output_file/pyarrow_output.parquet"
    pq.write_table(table, output_path)

    # Store output file path
    results["output_file"] = output_path

    # Step 4: Read the Parquet file back into Pandas (for preview/validation)
    df_parquet = pd.read_parquet(output_path)

    # Return both:
    # 1. Clickable file link (for download in Jupyter)
    # 2. Preview of stored data
    return FileLink(output_path), df_parquet.head()


# Call function
columnar_storage(df)

(c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\pyarrow_output.parquet,
     id       name  category   price quantity        date
 7    8  Product_H  Clothing  1499.0      ten  2024-02-22
 9   10  Product_J  Clothing  1144.0      NaN  2024-02-22
 13  14  Product_N   Grocery   868.0        6         NaN
 17  18  Product_R   Grocery  2343.0      NaN  2024-02-02
 19  20  Product_T  Clothing   627.0      ten  2024-01-03)

### Step 5.1.3: polars

In [9]:
def polars_problem1(input_path):
    """
    Read a CSV file, clean the 'price' column, perform aggregation,
    and save the result using Polars.

    This function performs:
    1. Reads input CSV using Polars
    2. Cleans 'price' column by removing non-numeric characters
    3. Converts 'price' to float (handles invalid values safely)
    4. Aggregates total revenue per 'id'
    5. Saves the result as a CSV file
    6. Returns the output file path

    Parameters:
    ----------
    input_path : str
        Path to the input CSV file

    Returns:
    -------
    str
        Path to the generated CSV file containing aggregated results
    """

    # Read CSV and clean 'price' column
    # - Remove any non-numeric characters (like currency symbols)
    # - Convert cleaned values to float
    df = pl.read_csv(input_path).with_columns(
        pl.col("price")
        .str.replace_all(r"[^0-9.]", "")
        .cast(pl.Float64, strict=False)
    )

    # Group by 'id' and calculate total revenue (sum of price)
    total_revenue = df.group_by("id").agg(
        pl.col("price").sum().alias("total_revenue")
    )

    # Ensure output directory exists
    output_dir = "../output_file"
    os.makedirs(output_dir, exist_ok=True)

    # Define output file path
    output_path = os.path.join(output_dir, "polars_aggregation.csv")

    # Write aggregated result to CSV
    total_revenue.write_csv(output_path)

    # Return output file path
    return output_path


# 🔥 EXECUTION (ONLY LINK SHOWN)
output_path = polars_problem1("../input_file/sample_500_rows.csv")
display(FileLink(output_path))

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\polars_aggregation.csv

### Step 5.1.4: DuckDB

In [ ]:
def duckdb_problem1(input_path):
    """
    Execute SQL on CSV using DuckDB, clean 'price', aggregate,
    save result to CSV, and return output file path.
    """

    query = f"""
        SELECT 
            category,
            COUNT(*) AS count,
            AVG(price_clean) AS avg_price
        FROM (
            SELECT 
                category,
                TRY_CAST(price AS DOUBLE) AS price_clean
            FROM read_csv_auto('{input_path}')
        )
        WHERE price_clean IS NOT NULL
        GROUP BY category
    """

    # Execute query
    result = duckdb.query(query).to_df()

    # Ensure output directory exists
    output_dir = "../output_file"
    os.makedirs(output_dir, exist_ok=True)

    # Output path
    output_path = os.path.join(output_dir, "duckdb_aggregation.csv")

    # Save result
    result.to_csv(output_path, index=False)

    return output_path


#  EXECUTION (ONLY LINK SHOWN)
output_path = duckdb_problem1("../input_file/sample_500_rows.csv")
display(FileLink(output_path))

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\duckdb_aggregation.csv

## Step 5.2: BIG DATA PROCESSING

### Step 5.2.1: pyspark

In [ ]:
'''Not configured as system tool'''
# def pyspark_problem1(input_path):
#     """
#     Optimized PySpark version:
#     - Faster startup
#     - Avoid unnecessary overhead
#     - Clean + aggregate
#     """

#     # Start Spark in local mode (important for speed)
#     spark = SparkSession.builder \
#         .appName("PySpark_Aggregation") \
#         .master("local[*]") \
#         .config("spark.sql.shuffle.partitions", "2") \
#         .getOrCreate()

#     # Reduce logs (optional but cleaner)
#     spark.sparkContext.setLogLevel("ERROR")

#     # Read CSV (avoid inferSchema → faster)
#     df = spark.read.option("header", True).csv(input_path)

#     # Clean price column
#     df_clean = df.withColumn(
#         "price_clean",
#         regexp_replace(col("price"), "[^0-9.]", "").cast("double")
#     ).filter(col("price_clean").isNotNull())

#     # Aggregation
#     result = df_clean.groupBy("category").agg(
#         avg("price_clean").alias("avg_price"),
#         sum("price_clean").alias("total_price")
#     )

#     # Output directory
#     output_dir = "../output_file"
#     os.makedirs(output_dir, exist_ok=True)

#     output_path = os.path.join(output_dir, "pyspark_aggregation")

#     # Save output (single file, minimal partitions)
#     result.coalesce(1).write.mode("overwrite").option("header", True).csv(output_path)

#     return output_path


# #  EXECUTION
# output_path = pyspark_problem1("../input_file/sample_500_rows.csv")
# display(FileLink(output_path))

### Step 5.2.2: Dask

In [ ]:

def dask_mean_problem(input_path):
    """
    Compute average price per category using Dask (parallel).
    Ensures 'price' is numeric before aggregation.
    """

    # Read CSV
    df = dd.read_csv(input_path)

    # REQUIRED: convert to numeric
    df["price"] = dd.to_numeric(df["price"], errors="coerce")

    # Drop invalid values (minimal cleaning)
    df = df.dropna(subset=["price"])

    # Compute mean
    summary = df.groupby("category").agg({
        "price": "mean"
    }).rename(columns={"price": "avg_price"})

    # Output path
    os.makedirs("../output_file", exist_ok=True)
    output_path = "../output_file/dask_output_file.csv"

    # Execute and save
    summary.compute().to_csv(output_path)

    return output_path


# EXECUTION
output_path = dask_mean_problem("../input_file/sample_500_rows.csv")
display(FileLink(output_path))

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\dask_output_file.csv

## Step 5.3: DATA VALIDATION

### Step 5.3.1: Great Expectations

In [ ]:


def ge_validation_only(input_path):
    """
    Great Expectations validation only
    """

    # Read CSV
    df = pd.read_csv(input_path)

    # Enforce type (CRITICAL)
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    # Validator
    validator = gx.validator.validator.Validator(
        execution_engine=gx.execution_engine.PandasExecutionEngine(),
        batches=[gx.core.batch.Batch(data=df)]
    )

    # Expectations
    validator.expect_column_values_to_not_be_null("price")
    validator.expect_column_values_to_be_between("price", min_value=0)
    validator.expect_column_values_to_not_be_null("category")
    validator.expect_column_values_to_not_be_in_set("category", [""])
    validator.expect_table_row_count_to_be_between(min_value=1)

    results = validator.validate()


    results_df = pd.DataFrame([{
        "expectation": r.expectation_config.type,
        "success": r.success
    } for r in results.results])

    # Save output
    os.makedirs("../output_file", exist_ok=True)
    output_path = "../output_file/great_expectations_validation.csv"
    results_df.to_csv(output_path, index=False)

    return output_path


# EXECUTION
output_path = ge_validation_only("../input_file/sample_500_rows.csv")
display(FileLink(output_path))

C:\Users\U1109200\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\great_expectations\expectations\expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 6/6 [00:00<00:00, 409.28it/s] 
C:\Users\U1109200\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\great_expectations\expectations\expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 538.52it/s] 
C:\Users\U1109200\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\great_expectations_validation.csv

### Step 5.3.2: Pandera

In [ ]:

def pandera_validation(input_path):
    """
    Robust Pandera validation

    - Does NOT modify input file
    - Handles string → numeric safely
    - Handles invalid indices properly
    - Splits clean + bad data
    - Saves inside /output_file/pandera/
    """

    # Read raw data 
    df = pd.read_csv(input_path)

    # Safe copy for validation
    df_copy = df.copy()
    df_copy["price"] = pd.to_numeric(df_copy["price"], errors="coerce")

    # Schema
    schema = pa.DataFrameSchema({
        "price": pa.Column(float, checks=pa.Check.gt(0), nullable=False),
        "category": pa.Column(str, checks=pa.Check.str_length(min_value=1), nullable=False)
    })

    try:
        validated_df = schema.validate(df_copy, lazy=True)
        clean_df = validated_df
        bad_df = pd.DataFrame()

    except pa.errors.SchemaErrors as err:
        failure_cases = err.failure_cases

        # Fix index issues properly
        bad_index = (
            failure_cases["index"]
            .dropna()
            .astype(int)
            .unique()
        )

        bad_df = df.loc[bad_index]
        clean_df = df.drop(bad_index)

    # Create nested folder
    output_dir = "../output_file/pandera"
    os.makedirs(output_dir, exist_ok=True)

    # Proper file paths
    clean_path = os.path.join(output_dir, "pandera_clean.csv")
    bad_path = os.path.join(output_dir, "pandera_invalid.csv")

    # Save files
    clean_df.to_csv(clean_path, index=False)
    bad_df.to_csv(bad_path, index=False)

    return clean_path, bad_path


# EXECUTION
clean, bad = pandera_validation("../input_file/sample_500_rows.csv")

display(FileLink(clean))
display(FileLink(bad))

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\pandera\pandera_clean.csv

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\pandera\pandera_invalid.csv

### Step 5.3.3: Pydantic

In [ ]:

class Product(BaseModel):
    name: str
    category: str
    price: float
    quantity: int | None = None
    date: str | None = None

    # Custom validation
    @field_validator("price")
    def price_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("price must be > 0")
        return v

    @field_validator("category")
    def category_not_empty(cls, v):
        if not v or v.strip() == "":
            raise ValueError("category cannot be empty")
        return v


def pydantic_validation(input_path):
    """
    Pydantic validation pipeline

    - Read CSV
    - Validate each row
    - Split into clean & invalid
    - Save inside /output_file/pydantic/
    """

    df = pd.read_csv(input_path)

    valid_rows = []
    invalid_rows = []

    for _, row in df.iterrows():
        try:
            product = Product(**row.to_dict())
            valid_rows.append(product.model_dump())

        except ValidationError as e:
            bad_row = row.to_dict()
            bad_row["error"] = str(e)
            invalid_rows.append(bad_row)

    # Convert to DataFrames
    clean_df = pd.DataFrame(valid_rows)
    bad_df = pd.DataFrame(invalid_rows)

    # Create nested folder
    output_dir = "../output_file/pydantic"
    os.makedirs(output_dir, exist_ok=True)

    # File paths inside folder
    clean_path = os.path.join(output_dir, "pydantic_clean.csv")
    bad_path = os.path.join(output_dir, "pydantic_invalid.csv")

    # Save files
    clean_df.to_csv(clean_path, index=False)
    bad_df.to_csv(bad_path, index=False)

    return clean_path, bad_path


# EXECUTION
clean, bad = pydantic_validation("../input_file/sample_500_rows.csv")

display(FileLink(clean))
display(FileLink(bad))

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\pydantic\pydantic_clean.csv

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\pydantic\pydantic_invalid.csv

## Step 5.4: DATA TRANSFORMATION

### Step 5.4.1: Pyjanitor

In [18]:
def pyjanitor_validation(input_path):
    """
    PyJanitor validation pipeline

    - Read CSV
    - Clean column names
    - Convert price safely
    - Split into clean & invalid rows
    - Save inside ../output_file/pyjanitor/
    """

    # Read CSV
    df = pd.read_csv(input_path)

    # Clean column names
    df = df.clean_names()

    # SAFE conversion (prevents Arrow errors)
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    # Validation logic
    invalid_mask = (
        df["price"].isna() |
        (df["price"] <= 0) |
        df["category"].isna() |
        (df["category"].astype(str).str.strip() == "")
    )

    # Split data
    clean_df = df[~invalid_mask]
    bad_df = df[invalid_mask].copy()

    # Add error column
    bad_df["error"] = "Invalid price or category"

    # ✅ Correct path (go OUT of code_file → into output_file)
    output_dir = os.path.join("..", "output_file", "pyjanitor")
    os.makedirs(output_dir, exist_ok=True)

    # File paths
    clean_path = os.path.join(output_dir, "pyjanitor_clean.csv")
    bad_path = os.path.join(output_dir, "pyjanitor_invalid.csv")

    # Save
    clean_df.to_csv(clean_path, index=False)
    bad_df.to_csv(bad_path, index=False)

    return clean_path, bad_path


# 🔥 EXECUTION
clean, bad = pyjanitor_validation("../input_file/sample_500_rows.csv")

display(FileLink(clean))
display(FileLink(bad))

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\pyjanitor\pyjanitor_clean.csv

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\pyjanitor\pyjanitor_invalid.csv

### Step 5.4.2: Toolz

In [21]:
# 🔧 Pipeline steps

def read_data(path):
    return pd.read_csv(path)


def clean_columns(df):
    df.columns = df.columns.str.lower().str.strip()
    return df


def convert_price(df):
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    return df


def split_data(df):
    invalid_mask = (
        df["price"].isna() |
        (df["price"] <= 0) |
        df["category"].isna() |
        (df["category"].astype(str).str.strip() == "")
    )

    clean_df = df[~invalid_mask]
    bad_df = df[invalid_mask].copy()
    bad_df["error"] = "Invalid price or category"

    return clean_df, bad_df


def save_output(clean_df, bad_df):
    output_dir = os.path.join("..", "output_file", "toolz")
    os.makedirs(output_dir, exist_ok=True)

    clean_path = os.path.join(output_dir, "toolz_clean.csv")
    bad_path = os.path.join(output_dir, "toolz_invalid.csv")

    clean_df.to_csv(clean_path, index=False)
    bad_df.to_csv(bad_path, index=False)

    return clean_path, bad_path


# 🔥 Main function
def toolz_validation(input_path):
    """
    Toolz pipeline

    - Functional transformations
    - Clean pipeline structure
    - Save inside ../output_file/toolz/
    """

    clean_df, bad_df = pipe(
        input_path,
        read_data,
        clean_columns,
        convert_price,
        split_data
    )

    return save_output(clean_df, bad_df)


# 🔥 EXECUTION
clean, bad = toolz_validation("../input_file/sample_500_rows.csv")

display(FileLink(clean))
display(FileLink(bad))

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\toolz\toolz_clean.csv

c:\Users\U1109200\Downloads\Dummy_rep-main\Libraries_implement\output_file\toolz\toolz_invalid.csv